# 练习作业 1：共享单车租车量预测（回归）

> 这份作业是照着李宏毅 HW1（COVID-19 阳性率预测）的骨架做的。
> 目的不是刷分，是让你把「改一个东西 → 跑一次 → 看结果 → 猜原因」这个循环多走几遍。

## 任务
根据时间、天气、节假日等信息，预测一个共享单车站点的租车量。

## 你会练到
- 用 PyTorch 写完整的回归训练流程
- 判断哪些特征是噪声，做特征选择
- 控制变量：一次只改一个设置，比较 Valid loss 和测试 RMSE
- 早停、保存最优模型、用保存的模型做预测

## 数据
- `bike_train.csv`：3200 行，55 个特征 + 最后 1 列 `target`
- `bike_test.csv`：1200 行，只有 55 个特征
- `test_labels.csv`：测试集真实答案，**只用来最后评分，不要拿去训练**

## 规则
1. **一次只改一个地方**，其余保持不变
2. 改了模型结构或特征后，**必须重新运行「创建模型」那一格**
3. 每个实验都记录：改了什么、最优 Valid loss、测试 RMSE、你的结论
4. 卡住先自己看 20 分钟：报错最后一行、张量形状、把中间结果 print 出来


## 0. 导入和路径

In [13]:
import math
import os
import csv

import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter


# 找到数据目录：无论 VS Code 的工作目录是哪里，都能找到
BASE = Path.cwd()
if not (BASE / "bike_train.csv").exists():
    candidate = BASE / "ML-Setup" / "Practice_1"
    if candidate.exists():
        BASE = candidate
assert (BASE / "bike_train.csv").exists(), f"找不到数据，请检查路径：{BASE}"
print("数据目录:", BASE)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


数据目录: c:\Users\35433\Desktop\Claude ML\ML-Setup\Practice_1
device: cuda


## 1. 先看一眼数据（养成习惯）

不要急着跑模型。先看看有多少行、列名长什么样、target 大概是什么范围。
这一步在 HW1 里就是「应该自己发现州名是噪声」的那一步。

In [14]:
df = pd.read_csv(BASE / "bike_train.csv")
print("形状:", df.shape)
print("\n列名:")
for i, name in enumerate(df.columns):
    print(f"  {i:2d}  {name}")
print("\ntarget 统计:")
print(df["target"].describe())
print("\n前 3 行:")
print(df.head(3).to_string())


形状: (3200, 56)

列名:
   0  region_01
   1  region_02
   2  region_03
   3  region_04
   4  region_05
   5  region_06
   6  region_07
   7  region_08
   8  region_09
   9  region_10
  10  region_11
  11  region_12
  12  region_13
  13  region_14
  14  region_15
  15  region_16
  16  region_17
  17  region_18
  18  region_19
  19  region_20
  20  noise_01
  21  noise_02
  22  noise_03
  23  noise_04
  24  noise_05
  25  noise_06
  26  noise_07
  27  noise_08
  28  noise_09
  29  noise_10
  30  hour
  31  month
  32  weekday
  33  workingday
  34  holiday
  35  is_weekend
  36  season
  37  temp
  38  feels_like
  39  humidity
  40  windspeed
  41  weather
  42  visibility
  43  dew_point
  44  pressure
  45  cloud_cover
  46  solar_radiation
  47  rainfall
  48  snowfall
  49  rush_hour
  50  daylight
  51  temp_humidity
  52  wind_chill
  53  heat_index
  54  comfort_index
  55  target

target 统计:
count    3200.000000
mean      138.268245
std        70.378571
min         0.000000
25%    

## 2. 工具函数（不用改）

In [15]:
def same_seed(seed):
    '''固定随机种子，让每次运行结果尽量可复现。'''
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def train_valid_split(data_set, valid_ratio, seed):
    '''把训练数据随机切成训练集和验证集。'''
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(data_set))
    valid_size = int(valid_ratio * len(data_set))
    valid_idx, train_idx = idx[:valid_size], idx[valid_size:]
    return data_set[train_idx], data_set[valid_idx]


def predict(test_loader, model, device):
    model.eval()
    preds = []
    for x in tqdm(test_loader):
        x = x.to(device)
        with torch.no_grad():
            pred = model(x)
            preds.append(pred.detach().cpu())
    return torch.cat(preds, dim=0).numpy()


def save_pred(preds, file):
    '''把预测结果存成 csv，格式是 id,target。'''
    with open(file, "w", newline="") as fp:
        writer = csv.writer(fp)
        writer.writerow(["id", "target"])
        for i, p in enumerate(preds):
            writer.writerow([i, float(p)])


## 3. Dataset（不用改）

把 numpy 数组包装成 PyTorch 能按索引取数的格式。

In [16]:
class BikeDataset(Dataset):
    '''
    x: 特征
    y: 目标值；如果传 None，说明是在做预测，没有答案
    '''
    def __init__(self, x, y=None):
        self.x = torch.FloatTensor(x)
        self.y = None if y is None else torch.FloatTensor(y)

    def __getitem__(self, idx):
        if self.y is None:
            return self.x[idx]
        return self.x[idx], self.y[idx]

    def __len__(self):
        return len(self.x)


## 4. 模型（可以改）

这是和 HW1 一样的两层全连接网络。想加宽/加深，改 `self.layers` 里的数字或加层。
注意：**最后一层输出必须是 1**，因为我们要预测一个数。

In [93]:
class My_Model(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        # TODO: 试试改宽度、深度，或者加 Dropout。
        self.layers = nn.Sequential(
            nn.Linear(input_dim,16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),  #改动第一次,16->72,8->36；第二个实验第一次改动72/36->36/18；第二次实验第二次更改，尝试正向加大增加力度36/18->108/54
            nn.Linear(8, 1),
        )

    def forward(self, x):
        # (B, 1) -> (B)，这样和 y 的形状对得上
        return self.layers(x).squeeze(1)


## 5. 特征选择（重点练习）

`select_all=True` 时用全部 55 列；`False` 时用 `feat_idx` 里选的列。

提示：数据被故意排成三段——
- 第 0–19 列：`region_xx`，站点区域 one-hot，**纯噪声**
- 第 20–29 列：`noise_xx`，**纯噪声**
- 第 30 列以后：时间、天气等**可能有用的特征**

先跑一次 `select_all=True` 当基线，再改成 `False` 看 Valid loss 和 RMSE 有没有变好。

In [83]:
def select_feat(train_data, valid_data, test_data, select_all=True):
    '''选择要送进模型的特征列。'''
    y_train, y_valid = train_data[:, -1], valid_data[:, -1]
    raw_x_train, raw_x_valid, raw_x_test = train_data[:, :-1], valid_data[:, :-1], test_data

    if select_all:
        feat_idx = list(range(raw_x_train.shape[1]))
    else:
        # TODO: 自己决定保留哪些列。
        # 下面这行是一个「跳过前 30 列噪声」的可用起点，先跑通，再试着改。
        feat_idx = list(range(30, raw_x_train.shape[1]))

    return (
        raw_x_train[:, feat_idx],
        raw_x_valid[:, feat_idx],
        raw_x_test[:, feat_idx],
        y_train,
        y_valid,
    )


## 6. 训练循环（优化器可以改）

默认用 `Adam`，因为对你的数据更容易跑出结果。
HW1 用的是 `SGD + momentum`，你也可以在 `<TODO>` 那里换成它做对比。

In [108]:
def trainer(train_loader, valid_loader, model, config, device):
    criterion = nn.MSELoss(reduction="mean")

    # TODO: 换优化器做对比，例如：
    optimizer = torch.optim.SGD(model.parameters(), lr=config["learning_rate"], momentum=0.9)
    # optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])

    writer = SummaryWriter(log_dir=str(BASE / "runs"))
    os.makedirs(os.path.dirname(config["save_path"]), exist_ok=True)

    n_epochs, best_loss, best_train_loss, step, early_stop_count = config["n_epochs"], math.inf, math.inf, 0, 0

    for epoch in range(n_epochs):
        model.train()
        loss_record = []
        train_pbar = tqdm(train_loader, position=0, leave=True)

        for x, y in train_pbar:
            optimizer.zero_grad()
            x, y = x.to(device), y.to(device)
            pred = model(x)
            loss = criterion(pred, y)
            loss.backward()
            optimizer.step()

            step += 1
            loss_record.append(loss.detach().item())
            train_pbar.set_description(f"Epoch [{epoch + 1}/{n_epochs}]")
            train_pbar.set_postfix({"loss": loss.detach().item()})

        mean_train_loss = sum(loss_record) / len(loss_record)
        writer.add_scalar("Loss/train", mean_train_loss, step)

        model.eval()
        loss_record = []
        with torch.no_grad():
            for x, y in valid_loader:
                x, y = x.to(device), y.to(device)
                pred = model(x)
                loss = criterion(pred, y)
                loss_record.append(loss.item())

        mean_valid_loss = sum(loss_record) / len(loss_record)
        writer.add_scalar("Loss/valid", mean_valid_loss, step)

        if (epoch + 1) % 20 == 0:
            print(
                f"Epoch [{epoch + 1}/{n_epochs}]: "
                f"Train loss: {mean_train_loss:.2f}, Valid loss: {mean_valid_loss:.2f}"
            )

        if mean_valid_loss < best_loss:
            best_loss = mean_valid_loss
            best_train_loss = mean_train_loss      # 记下最优那一轮的 Train loss
            torch.save(model.state_dict(), config["save_path"])
            print(f"  -> 新的最优 Valid loss: {best_loss:.2f}（已保存）")
            early_stop_count = 0
        else:
            early_stop_count += 1

        if early_stop_count >= config["early_stop"]:
            print(f"\n连续 {config['early_stop']} 轮没有进步，停止训练。")
            print(f"最优 Valid loss = {best_loss:.2f}，该轮 Train loss = {best_train_loss:.2f}")
            return

    print(f"\n训练结束，最优 Valid loss = {best_loss:.2f}，该轮 Train loss = {best_train_loss:.2f}")


## 7. 配置

所有超参数集中在这里。先跑默认值，之后每次只改一个。

In [85]:
config = {
    "seed": 2026,
    "select_all": False,      # TODO: 基线先 True；再改 False 做特征选择实验
    "valid_ratio": 0.2,
    "n_epochs": 3000,
    "batch_size": 256,
    "learning_rate": 1e-3,
    "early_stop": 300,
    "save_path": str(BASE / "models" / "model.ckpt"),
}
config


{'seed': 2026,
 'select_all': False,
 'valid_ratio': 0.2,
 'n_epochs': 3000,
 'batch_size': 256,
 'learning_rate': 0.001,
 'early_stop': 300,
 'save_path': 'c:\\Users\\35433\\Desktop\\Claude ML\\ML-Setup\\Practice_1\\models\\model.ckpt'}

## 8. 读取数据 + DataLoader

In [103]:
same_seed(config["seed"])

train_data = pd.read_csv(BASE / "bike_train.csv").values
test_data = pd.read_csv(BASE / "bike_test.csv").values
train_data, valid_data = train_valid_split(train_data, config["valid_ratio"], config["seed"])

print(f"train_data size: {train_data.shape}")
print(f"valid_data size: {valid_data.shape}")
print(f"test_data size:  {test_data.shape}")

x_train, x_valid, x_test, y_train, y_valid = select_feat(
    train_data, valid_data, test_data, config["select_all"]
)
print(f"number of features: {x_train.shape[1]}")

# 只标准化 target：减训练集均值、除训练集标准差
# 之后训练打印的 Valid loss 是“标准化单位”（约 0.1），不能和之前的 585.65 直接比
y_mean = y_train.mean()
y_std = y_train.std()
y_train = (y_train - y_mean) / y_std
y_valid = (y_valid - y_mean) / y_std
print(f"target 已标准化: mean={y_mean:.2f}, std={y_std:.2f}")

train_dataset = BikeDataset(x_train, y_train)
valid_dataset = BikeDataset(x_valid, y_valid)
test_dataset = BikeDataset(x_test)

train_loader = DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=config["batch_size"], shuffle=False, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=config["batch_size"], shuffle=False, pin_memory=True)


train_data size: (2560, 56)
valid_data size: (640, 56)
test_data size:  (1200, 55)
number of features: 25
target 已标准化: mean=138.73, std=70.65


## ⚠️ 改完东西要重跑哪些格

| 你改了什么 | 必须重跑 |
|---|---|
| 模型结构（`My_Model`） | 第 9 节（创建模型 + 训练） |
| `trainer` 里的优化器 | 第 6 节（重新定义 trainer）+ 第 9 节 |
| `select_all` / `feat_idx` / `valid_ratio` / `seed` | 第 8 节（DataLoader）+ 第 9 节 |
| `learning_rate` / `n_epochs` / `early_stop` | 第 7 节（config）+ 第 9 节 |
| 数据文件本身 | 从第 8 节开始 |

为什么：

- 第 8 节会把 `config["select_all"]` 传进 `select_feat`，生成新的 `x_train`。只改 `config` 不重跑第 8 节，特征选择不会生效。
- 第 9 节才会 `My_Model(...)` 创建**全新的随机模型**。不重跑第 9 节，就是在上一轮的模型上继续训，实验结论会失真。


## 9. 开始训练

改过模型/特征/配置后，**这一格必须整格重新运行**，它才会创建一个全新的随机模型。

In [109]:
model = My_Model(input_dim=x_train.shape[1]).to(device)
trainer(train_loader, valid_loader, model, config, device)


Epoch [1/3000]: 100%|██████████| 10/10 [00:00<00:00, 150.38it/s, loss=1.08]


  -> 新的最优 Valid loss: 0.96（已保存）


Epoch [2/3000]: 100%|██████████| 10/10 [00:00<00:00, 212.63it/s, loss=0.9]


  -> 新的最优 Valid loss: 0.93（已保存）


Epoch [3/3000]: 100%|██████████| 10/10 [00:00<00:00, 283.91it/s, loss=0.941]


...

Epoch [2320/3000]: Train loss: 0.08, Valid loss: 0.11


Epoch [2340/3000]: 100%|██████████| 10/10 [00:00<00:00, 199.60it/s, loss=0.0789]


Epoch [2340/3000]: Train loss: 0.08, Valid loss: 0.11


Epoch [2341/3000]: 100%|██████████| 10/10 [00:00<00:00, 300.11it/s, loss=0.0859]


连续 300 轮没有进步，停止训练。
最优 Valid loss = 0.11，该轮 Train loss = 0.08


## 10. 预测并保存

训练时每次 Valid loss 变好都会存一份最优模型，这里把它读回来做预测。

In [110]:
model = My_Model(input_dim=x_train.shape[1]).to(device)
model.load_state_dict(torch.load(config["save_path"], map_location=device))
preds = predict(test_loader, model, device)
preds = preds * y_std + y_mean      # 换回原始单位
save_pred(preds, str(BASE / "pred.csv"))
print("预测已保存到:", BASE / "pred.csv")


100%|██████████| 5/5 [00:00<00:00, 857.66it/s]

预测已保存到: c:\Users\35433\Desktop\Claude ML\ML-Setup\Practice_1\pred.csv


## 11. 本地评分

`test_labels.csv` 是测试集真实答案，只在这里用。
最后一行的「全猜训练集平均值」是最笨的基线——你的 RMSE 必须比它低，否则模型没学到东西。

In [111]:
pred_df = pd.read_csv(BASE / "pred.csv")
label_df = pd.read_csv(BASE / "test_labels.csv")

y_true = label_df["target"].values
y_pred = pred_df["target"].values

mse = np.mean((y_true - y_pred) ** 2)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(y_true - y_pred))
r2 = 1 - np.sum((y_true - y_pred) ** 2) / np.sum((y_true - y_true.mean()) ** 2)

train_mean = pd.read_csv(BASE / "bike_train.csv")["target"].mean()
baseline_rmse = np.sqrt(np.mean((y_true - train_mean) ** 2))

print(f"MSE  = {mse:.2f}")
print(f"RMSE = {rmse:.2f}   (越小越好)")
print(f"MAE  = {mae:.2f}")
print(f"R^2  = {r2:.4f}")
print(f"「全猜训练集平均值」的 RMSE = {baseline_rmse:.2f}")
print(f"你比这个笨基线好了 {baseline_rmse - rmse:.2f}")


MSE  = 546.60
RMSE = 23.38   (越小越好)
MAE  = 18.56
R^2  = 0.8876
「全猜训练集平均值」的 RMSE = 69.76
你比这个笨基线好了 46.38


## 12. 实验清单（至少做 4 组）

| 编号 | 改什么 | 其他设置 | 最优 Valid loss | 测试 RMSE | 结论 |
|---|---|---|---|---|---|
| E1 | 基线：`select_all=True` | 默认模型 + Adam 1e-3 | | | |
| E2 | `select_all=False`（去掉噪声列） | 其他不变 | | | |
| E3 | 模型加宽：16/8 → 64/32 | 在 E2 基础上 | | | |
| E4 | 优化器换成 SGD + momentum | 在 E2 基础上 | | | |
| E5（可选） | 加 `weight_decay` 或 Dropout | | | | |

规则：

1. **一次只改一个变量**，否则说不清是谁起的作用（HW1 就吃过这个亏）
2. 改完 `config` 或模型后，重新运行「创建模型并训练」那一格
3. 每组实验前先写下你的猜测：会变好还是变差？为什么？
4. 训练结束后把两个数字记下来：最优 Valid loss、测试 RMSE


## 13. 记录你的实验结果

把每次的结果 append 进来，最后 `pd.DataFrame(results)` 就是一张对比表。

In [ ]:
results=[
    {
        "实验": "E1 基线",
        "改了什么": "什么都没改（select_all=True，默认模型，Adam 1e-3）",
        "Training Loss":392.67,
        "最优 Valid loss": 684.94,
        "测试 RMSE": 24.45,
        "结论": "对照组。后面所有实验都跟它比",
    },
    
    
    # 第一次实验，去掉列表噪声，猜测模型训练变好
    {
        "实验": "E2 改成False",
        "改了什么": "（select_all=False，去掉噪声列）",
        "Training Loss":440.94,
        "最优 Valid loss": 585.65,
        "测试 RMSE": 23.10,
        "结论": "去除噪声以后模型训练变好",
    },
    
    
    
    #第二次实验，把模型加宽，16->72，8->36，预测加宽以后时间变长但是效果变好
    {
        "实验": "把模型加宽，16->72，8->36",
        "改了什么": "（select_all=False没有动，把模型加宽，16->72，8->36）",
        "Training Loss":300.77,
        "最优 Valid loss": 596.27,
        "测试 RMSE": 24.04,
        "结论": "模型加宽以后RMSE反而更大了，是不是加宽程度太大，进行二次试验",
    },
    
    #第二次实验洗一次更改，收敛一下模型加宽，达到更好的效果，72->36，36->18
    {
        "实验": "把模型加宽收敛，72->36，36->18",
        "改了什么": "（select_all=False没有动，把模型加宽收敛，72->36，36->18）",
        "Training Loss":341.40,
        "最优 Valid loss": 633.35,
        "测试 RMSE": 23.87,
        "结论": "模型加宽收敛以后RMSE变小但是效果不是很明显，尝试正向增大加宽程度，进行三次试验",
    },
    
    #第二次实验第二次更改，尝试正向加大增加力度36/18->108/54
    {
        "实验": "把模型加宽收敛，72->36，36->18",
        "改了什么": "（select_all=False没有动，正向加大增加力度36/18->108/54）",
        "Training Loss":310.63,
        "最优 Valid loss": 590.48,
        "测试 RMSE": 24.40,
        "结论": "模型加宽正向增大以后Valid Loss更大，原因是参数量太大了，出现了过拟合",
    },
    
    #第二次实验第三次更改，尝试修正宽度为108/54->64/32
    {
        "实验": "把模型加宽修正宽度为108/54->64/32",
        "改了什么": "（select_all=False没有动，修正宽度为108/54->64/32）",
        "Training Loss":315.38,
        "最优 Valid loss": 604.89,
        "测试 RMSE": 24.27,
        "结论": "改变宽度不可取，尝试更换优化器",
    },

    #第三次实验，改变优化器
    {
        "实验": "E4 SGD + momentum",
        "改了什么": "Adam → SGD(momentum=0.9)，lr 1e-3，target 未标准化，16/8，25 特征",
        "Training Loss":5006.32,
        "最优 Valid loss": 4755.93,
        "测试 RMSE": 69.03,
        "结论": "	欠拟合/未收敛：模型停在预测均值。SGD 单一学习率与未标准化的 target 尺度不匹配；仅调 lr 无效。下一步标准化 target 再比较",
    },

    #第四次实验，target标准化，用SGD
    {
        "实验": "E5 target 标准化",
        "改了什么": "target 用训练集均值/标准差标准化，SGD",
        "Training Loss": 406.83,
        "最优 Valid loss": 549.12,
        "测试 RMSE": 23.38,
        "结论": "target 标准化后 SGD/Adam 的梯度尺度正常，模型能学特征了。Train 394 仍在噪声底线附近，Valid 527，测试 RMSE 22.62，是目前最好的一组（比 E2 好 0.48）",
    },

    #第五次实验，target标准化，用Adam
    {
        "实验": "E6 target 标准化",
        "改了什么": "target 用训练集均值/标准差标准化，Adam 1e-3，模型 16/8，25 特征",
        "Training Loss": 394.36,
        "最优 Valid loss": 526.64,
        "测试 RMSE": 22.62,
        "结论": "target 标准化后 SGD/Adam 的梯度尺度正常，模型能学特征了。Train 394 仍在噪声底线附近，Valid 527，测试 RMSE 22.62，是目前最好的一组（比 E2 好 0.48）",
    }
]
pd.DataFrame(results)


,实验,改了什么,Training Loss,最优 Valid loss,测试 RMSE,结论
0,E1 基线,什么都没改（select_all=True，默认模型，Adam 1e-3）,392.67,684.94,24.45,对照组。后面所有实验都跟它比
1,E2 改成False,（select_all=False，去掉噪声列）,440.94,585.65,23.10,去除噪声以后模型训练变好
2,把模型加宽，16->72，8->36,（select_all=False没有动，把模型加宽，16->72，8->36）,300.77,596.27,24.04,模型加宽以后RMSE反而更大了，是不是加宽程度太大，进行二次试验
3,把模型加宽收敛，72->36，36->18,（select_all=False没有动，把模型加宽收敛，72->36，36->18）,341.40,633.35,23.87,模型加宽收敛以后RMSE变小但是效果不是很明显，尝试正向增大加宽程度，进行三次试验
4,把模型加宽收敛，72->36，36->18,（select_all=False没有动，正向加大增加力度36/18->108/54）,310.63,590.48,24.40,模型加宽正向增大以后Valid Loss更大，原因是参数量太大了，出现了过拟合
5,把模型加宽修正宽度为108/54->64/32,（select_all=False没有动，修正宽度为108/54->64/32）,315.38,604.89,24.27,改变宽度不可取，尝试更换优化器
6,\tE4 SGD + momentum,\tAdam → SGD(momentum=0.9)，lr 1e-3，target 未标准化...,5006.32,4755.93,69.03,\t欠拟合/未收敛：模型停在预测均值。SGD 单一学习率与未标准化的 target 尺度不匹...
7,E5 target 标准化,target 用训练集均值/标准差标准化，Adam 1e-3，模型 16/8，25 特征,0.08,0.10,22.62,\t欠拟合/未收敛：模型停在预测均值。SGD 单一学习率与未标准化的 target 尺度不匹...


## 14. 交作业前检查

- [ ] 从头到尾按顺序跑通一遍，没有报错
- [ ] `pred.csv` 已生成，`id` 从 0 到 1199
- [ ] 至少完成 E1–E4 四组实验，结果填进上面的表
- [ ] 能用自己的话说出：去掉噪声特征后为什么变好/变差
- [ ] 能解释 `config["select_all"]` 是怎么控制 `select_feat` 走哪个分支的
- [ ] 能解释为什么改完模型要重新运行「创建模型」那一格

做完可以叫我，我帮你看结果、挑毛病。
